# 03b | Aggregation: GROUP BY, HAVING, CASE WHEN

The SQL counterpart of pandas' `groupby().agg()`.

**After this page you can:** compute grouped statistics, filter the grouped results, and use `CASE WHEN` to bucket values and compute proportions. About 45 minutes.

## 1. Aggregate functions

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. Without `GROUP BY`, the whole table is summarized into one row.

`COUNT` has three forms with different meanings, a common interview question:

| Form | What it counts |
|---|---|
| `COUNT(*)` | Rows |
| `COUNT(country)` | Rows where `country` is **not NULL** |
| `COUNT(DISTINCT country)` | Number of distinct `country` values (NULL not counted) |

In [ ]:
SELECT COUNT(*)                AS n_rows,
       COUNT(country)          AS n_with_country,
       COUNT(DISTINCT country) AS n_countries
FROM users;

## 2. GROUP BY

```sql
SELECT dept, COUNT(*) AS n, AVG(salary) AS avg_salary
FROM employees
GROUP BY dept;
```

Rule: **every column in `SELECT` must either appear in `GROUP BY` or be wrapped in an aggregate function.** The reason is simple: each group produces one row, and for a column that is neither grouped nor aggregated there is no way to know which value to take.

In [ ]:
SELECT dept, COUNT(*) AS n, AVG(salary) AS avg_salary, MAX(salary) AS max_salary
FROM employees
GROUP BY dept;

### ✏️ Exercise 1 (fill in the blanks)

Compute each user's number of orders and total order amount. Output `user_id`, `n_orders`, `total`.

In [ ]:
SELECT user_id,
       ____(*)      AS n_orders,
       ____(amount) AS total
FROM orders
GROUP BY ____;

<details>
<summary><b>Answer</b></summary>

```sql
SELECT user_id,
       COUNT(*)    AS n_orders,
       SUM(amount) AS total
FROM orders
GROUP BY user_id;
```

</details>

## 3. WHERE vs. HAVING

- `WHERE` filters **rows** **before** grouping;
- `HAVING` filters **groups** **after** grouping, so it can use aggregate functions.

"Completed orders only" filters rows, so use `WHERE`; "users with a total of at least 100" filters groups, so use `HAVING`.

In [ ]:
SELECT user_id, SUM(amount) AS total
FROM orders
WHERE status = 'completed'
GROUP BY user_id
HAVING SUM(amount) >= 100;

## 4. Execution order

The order in which the database **actually executes** the clauses differs from the order you write them:

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

This explains many "why does this error?" moments:

- You cannot use aggregate functions in `WHERE`: grouping has not happened yet when `WHERE` runs;
- You cannot use a `SELECT` alias in `WHERE`: `SELECT` has not run yet;
- You can use aliases in `ORDER BY`: it runs after `SELECT`.

### ✏️ Exercise 2 (write it yourself)

Find users with **at least 2 orders** (any status). Output `user_id`, `n_orders`, sorted by `n_orders` descending, with ties by `user_id` ascending.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT user_id, COUNT(*) AS n_orders
FROM orders
GROUP BY user_id
HAVING COUNT(*) >= 2
ORDER BY n_orders DESC, user_id;
```

</details>

## 5. CASE WHEN

SQL's if / elif / else, like `np.where` or R's `case_when`:

```sql
CASE
  WHEN condition1 THEN value1
  WHEN condition2 THEN value2
  ELSE value3
END
```

Conditions are checked top to bottom, and the first one that holds wins.

In [ ]:
SELECT order_id, amount,
       CASE
         WHEN amount < 20 THEN 'low'
         WHEN amount < 40 THEN 'mid'
         ELSE 'high'
       END AS bucket
FROM orders
LIMIT 6;

### ✏️ Exercise 3 (write it yourself)

Put all orders into three amount buckets (below 20 is `'low'`, 20 to 39 is `'mid'`, 40 and above is `'high'`) and count the orders in each. Output `bucket`, `n`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

Put the `CASE ... END AS bucket` from the example above into `SELECT`, then `GROUP BY bucket` and add `COUNT(*) AS n`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT CASE
         WHEN amount < 20 THEN 'low'
         WHEN amount < 40 THEN 'mid'
         ELSE 'high'
       END AS bucket,
       COUNT(*) AS n
FROM orders
GROUP BY bucket;
```

SQLite, MySQL and PostgreSQL all allow the alias `bucket` directly in `GROUP BY`. Some databases (such as SQL Server) do not; there you have to repeat the whole `CASE ... END`, or use a subquery (next page).

</details>

## 6. Conditional aggregation: CASE WHEN inside an aggregate

This is the single most useful trick in SQL interviews. To get "total orders" and "how many of them were refunded" in the same row:

```sql
SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END)
```

Each row scores 1 if it is a refund and 0 otherwise; the sum is the number of refunds. Replace `SUM` with `AVG` and you get the refund **rate**. This is the same as `sum()` and `mean()` of a boolean column in pandas.

⚠️ **Integer division:** in SQLite and PostgreSQL, `1 / 2` is `0` (dividing integers keeps only the integer part). When computing a ratio, multiply by `1.0` first: `1.0 * a / b`.

In [ ]:
SELECT COUNT(*)                                                   AS n_orders,
       SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END)       AS n_refunded,
       AVG(CASE WHEN status = 'refunded' THEN 1.0 ELSE 0 END)     AS refund_rate,
       SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_amount
FROM orders;

### ✏️ Exercise 4 (write it yourself)

For each user who placed an order, compute the total number of orders `n_orders`, the number of completed orders `n_completed`, and the total amount of completed orders `completed_total`. Output `user_id` and these three columns.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

Put all three aggregates in one `SELECT` with `GROUP BY user_id`. The last two use `SUM(CASE WHEN status = 'completed' THEN ... ELSE 0 END)`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT user_id,
       COUNT(*)                                                   AS n_orders,
       SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END)      AS n_completed,
       SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_total
FROM orders
GROUP BY user_id;
```

</details>

**🗣 Say it out loud:** What is the difference between WHERE and HAVING?

<details>
<summary><b>Reference answer</b></summary>

`WHERE` filters rows before grouping and cannot use aggregate functions; `HAVING` filters groups after aggregation and can use them. Any condition that can be applied early belongs in `WHERE`, so less data goes into the grouping.

**Short version:** WHERE filters rows before aggregation and can't reference aggregates; HAVING filters groups after aggregation. I put row-level conditions in WHERE so less data gets grouped.

</details>

## Summary

- `COUNT(*)` counts rows, `COUNT(col)` skips NULLs, `COUNT(DISTINCT col)` counts distinct values;
- Every column in `SELECT` is either in `GROUP BY` or aggregated;
- Filter rows with `WHERE`, groups with `HAVING`;
- `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` counts rows that meet a condition;
- Multiply by `1.0` before dividing to get a ratio.

On to **03c**.